# Ames Housing – K-means clustering

Dit notebook volgt de opdracht stap voor stap.  
De code is bewust simpel gehouden en sluit aan op de les over **K-means clustering**.

## Logger starten

We gebruiken een eenvoudige logger om belangrijke keuzes en uitkomsten vast te leggen.  
De logger schrijft mee in het notebook en in het bestand **clustering_logboek.txt**.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import logging

# Logger instellen
logger = logging.getLogger("clustering_logger")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(levelname)s - %(message)s")

file_handler = logging.FileHandler("clustering_logboek.txt", mode="w", encoding="utf-8")
stream_handler = logging.StreamHandler()

file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(stream_handler)

logger.info("Logger gestart.")

INFO - Logger gestart.


## Stap 1 – Inlezen dataset

We lezen het tabblad **AmesHousing** in als DataFrame.  
Daarna bekijken we kort de eerste rijen, de info en de kolomnamen.

In [ ]:
from pathlib import Path

bestand = Path("AmesHousing.xlsx")
if not bestand.exists():
    bestand = Path("/mnt/data/AmesHousing.xlsx")

sheet_naam = "AmesHousing"

df = pd.read_excel(bestand, sheet_name=sheet_naam)

logger.info(f"Bestand succesvol ingelezen: {bestand} ({df.shape[0]} rijen, {df.shape[1]} kolommen).")

df.head()

In [ ]:
df.info()

In [ ]:
df.columns

## Stap 2 – Korte verkenning

Hier bekijken we kort welke kolommen beschikbaar zijn en welke datatypes daarbij horen.  
We houden dit bewust klein en overzichtelijk.

In [ ]:
df.dtypes

## Stap 3 – Keuze van top 3 features

Volgens het tabblad **Data Dictionary** zijn dit voor een eerste model logische features om gelijkenissen tussen huizen te vinden:

1. **Overall Qual** – zegt iets over de algemene kwaliteit van het huis.  
2. **Gr Liv Area** – zegt iets over de woonoppervlakte boven de grond.  
3. **Neighborhood** – zegt in welke wijk het huis ligt. Dit is ook onze categorische feature.

Deze keuze is goed verdedigbaar voor clustering, omdat kwaliteit, grootte en ligging vaak veel zeggen over hoe vergelijkbaar huizen zijn.

In [ ]:
data_dictionary = pd.read_excel(bestand, sheet_name="Data Dictionary")
gekozen_features = ["Overall Qual", "Gr Liv Area", "Neighborhood"]

data_dictionary[data_dictionary["Variabele"].isin(gekozen_features)]

In [ ]:
logger.info(f"Gekozen top 3 features: {gekozen_features}")

## Stap 4 – Data prepareren

We maken een DataFrame met alleen de gekozen features.  
Daarna vullen we eventuele missende waarden simpel op, omdat **K-means niet met lege waarden kan werken**.  
Vervolgens zetten we de categorische feature om naar dummies met `pd.get_dummies()`.


In [ ]:
data_stap4 = df[gekozen_features].copy()

# KMeans kan niet werken met lege waarden, dus die vullen we eerst op.
data_stap4["Neighborhood"] = data_stap4["Neighborhood"].fillna("Onbekend")
data_stap4 = data_stap4.fillna(0)

# Dummies zetten tekstcategorieën om in 0/1-kolommen.
data_stap4_encoded = pd.get_dummies(data_stap4, columns=["Neighborhood"])

logger.info(f"One-hot encoding uitgevoerd. Nieuwe vorm van de data: {data_stap4_encoded.shape}")

data_stap4_encoded.head()

## Stap 5 – Eerste KMeans-model

Bij K-means is schalen vaak handig, omdat grote getallen anders zwaarder meetellen in de afstandsberekening.  
Daarom gebruiken we hier `StandardScaler()`.

Daarna bekijken we met `help(KMeans)` welke hyperparameters er zijn.

In [ ]:
scaler = StandardScaler()
data_stap5_scaled = scaler.fit_transform(data_stap4_encoded)

logger.info("Data geschaald met StandardScaler.")

### Hyperparameters bekijken

In [ ]:
help(KMeans)

### Eerste model trainen

Voor de eerste run kiezen we:
- **n_clusters = 4**
- **random_state = 42**
- **n_init = 10**

`n_clusters` is het aantal clusters.  
`n_init` is een extra hyperparameter: het model probeert dan meerdere startpunten en kiest de beste uitkomst.

In [ ]:
kmeans_1 = KMeans(n_clusters=4, random_state=42, n_init=10)

labels_1 = kmeans_1.fit_predict(data_stap5_scaled)

resultaat_1 = data_stap4.copy()
resultaat_1["Cluster"] = labels_1

logger.info("Eerste KMeans-model getraind.")
logger.info("Configuratie eerste model: n_clusters=4, random_state=42, n_init=10")

resultaat_1.head()

## Stap 6 – Evaluatie van het eerste model

We evalueren het model met:
- **Inertia / WSS**: hoe dicht de punten gemiddeld bij hun eigen clustercentrum liggen. Lager is meestal beter.
- **Silhouette score**: hoe goed clusters van elkaar gescheiden zijn. Hoger is meestal beter.

Daarnaast bekijken we de verdeling van huizen per cluster.

In [ ]:
inertia_1 = kmeans_1.inertia_
silhouette_1 = silhouette_score(data_stap5_scaled, labels_1)
cluster_verdeling_1 = resultaat_1["Cluster"].value_counts().sort_index()

print("Inertia / WSS:", inertia_1)
print("Silhouette score:", silhouette_1)
print()
print("Verdeling per cluster:")
print(cluster_verdeling_1)

logger.info(f"Eerste model - inertia: {inertia_1:.2f}")
logger.info(f"Eerste model - silhouette score: {silhouette_1:.4f}")
logger.info(f"Eerste model - clusterverdeling: {cluster_verdeling_1.to_dict()}")

### Korte interpretatie

- Een **lagere inertia** betekent dat huizen binnen een cluster dichter bij hun clustercentrum liggen.  
- Een **hogere silhouette score** betekent dat clusters beter van elkaar te onderscheiden zijn.  

Bij het vergelijken van experimenten is de silhouette score meestal het duidelijkst.  
De inertia is vooral handig als je modellen vergelijkt op **dezelfde voorbereide dataset**.

## Extra – Elleboogmethode

Met de elleboogmethode testen we meerdere waarden voor **k**.
We berekenen telkens de inertia/WSS en zoeken het punt waar de lijn **afvlakt**. Dat is vaak een logisch aantal clusters.

In [ ]:
k_waardes = range(2, 11)
wss_scores = []

for k in k_waardes:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(data_stap5_scaled)
    wss_scores.append(model.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_waardes), wss_scores, marker="o")
plt.xlabel("Aantal clusters (k)")
plt.ylabel("Inertia / WSS")
plt.title("Elleboogmethode")
plt.grid(True)
plt.show()

## Stap 7 – Experimenteren

Hieronder staan meerdere experimenten.  
Per experiment noteren we:
- gekozen features,
- categorische features,
- of er geschaald is,
- hyperparameters,
- inertia,
- silhouette score.

We bewaren alle werkende experimenten, zodat de ontwikkeling van het model zichtbaar blijft.

In [ ]:
experiment_resultaten = []

experiment_resultaten.append({
    "Experiment": "Initiële run",
    "Features": ", ".join(gekozen_feaddtures),
    "Categorische features": "Neighborhood",
    "Geschaald": "Ja",
    "n_clusters": 4,
    "n_init": 10,
    "Inertia": inertia_1,
    "Silhouette score": silhouette_1
})

### Experiment 1 – Zelfde features, maar ander aantal clusters

Hier houden we dezelfde features, maar kiezen we **5 clusters** in plaats van 4.  
Zo testen we of een ander aantal clusters beter past bij deze data.

In [ ]:
features_exp1 = ["Overall Qual", "Gr Liv Area", "Neighborhood"]
data_exp1 = df[features_exp1].copy()
data_exp1["Neighborhood"] = data_exp1["Neighborhood"].fillna("Onbekend")
data_exp1 = data_exp1.fillna(0)

data_exp1_encoded = pd.get_dummies(data_exp1, columns=["Neighborhood"])

scaler_exp1 = StandardScaler()
data_exp1_scaled = scaler_exp1.fit_transform(data_exp1_encoded)

kmeans_exp1 = KMeans(n_clusters=5, random_state=42, n_init=10)
labels_exp1 = kmeans_exp1.fit_predict(data_exp1_scaled)

inertia_exp1 = kmeans_exp1.inertia_
silhouette_exp1 = silhouette_score(data_exp1_scaled, labels_exp1)

print("Inertia / WSS:", inertia_exp1)
print("Silhouette score:", silhouette_exp1)

logger.info("Experiment 1 uitgevoerd.")
logger.info(f"Experiment 1 - features: {features_exp1}")
logger.info("Experiment 1 - categorische features: Neighborhood")
logger.info("Experiment 1 - geschaald: Ja")
logger.info("Experiment 1 - hyperparameters: n_clusters=5, random_state=42, n_init=10")
logger.info(f"Experiment 1 - inertia: {inertia_exp1:.2f}")
logger.info(f"Experiment 1 - silhouette score: {silhouette_exp1:.4f}")

experiment_resultaten.append({
    "Experiment": "Experiment 1 - ander aantal clusters",
    "Features": ", ".join(features_exp1),
    "Categorische features": "Neighborhood",
    "Geschaald": "Ja",
    "n_clusters": 5,
    "n_init": 10,
    "Inertia": inertia_exp1,
    "Silhouette score": silhouette_exp1
})

### Experiment 2 – Meer features toevoegen

Hier voegen we **Total Bsmt SF** toe.  
Zo testen we of extra informatie over grootte helpt bij het maken van duidelijkere clusters.

In [ ]:
features_exp2 = ["Overall Qual", "Gr Liv Area", "Total Bsmt SF", "Neighborhood"]
data_exp2 = df[features_exp2].copy()
data_exp2["Neighborhood"] = data_exp2["Neighborhood"].fillna("Onbekend")
data_exp2 = data_exp2.fillna(0)

data_exp2_encoded = pd.get_dummies(data_exp2, columns=["Neighborhood"])

scaler_exp2 = StandardScaler()
data_exp2_scaled = scaler_exp2.fit_transform(data_exp2_encoded)

kmeans_exp2 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_exp2 = kmeans_exp2.fit_predict(data_exp2_scaled)

inertia_exp2 = kmeans_exp2.inertia_
silhouette_exp2 = silhouette_score(data_exp2_scaled, labels_exp2)

print("Inertia / WSS:", inertia_exp2)
print("Silhouette score:", silhouette_exp2)

logger.info("Experiment 2 uitgevoerd.")
logger.info(f"Experiment 2 - features: {features_exp2}")
logger.info("Experiment 2 - categorische features: Neighborhood")
logger.info("Experiment 2 - geschaald: Ja")
logger.info("Experiment 2 - hyperparameters: n_clusters=4, random_state=42, n_init=10")
logger.info(f"Experiment 2 - inertia: {inertia_exp2:.2f}")
logger.info(f"Experiment 2 - silhouette score: {silhouette_exp2:.4f}")

experiment_resultaten.append({
    "Experiment": "Experiment 2 - extra feature",
    "Features": ", ".join(features_exp2),
    "Categorische features": "Neighborhood",
    "Geschaald": "Ja",
    "n_clusters": 4,
    "n_init": 10,
    "Inertia": inertia_exp2,
    "Silhouette score": silhouette_exp2
})

### Experiment 3 – Andere categorische feature

Hier gebruiken we **House Style** in plaats van **Neighborhood**.  
Zo testen we of woningtype beter helpt dan wijk.

In [ ]:
features_exp3 = ["Overall Qual", "Gr Liv Area", "House Style"]
data_exp3 = df[features_exp3].copy()
data_exp3["House Style"] = data_exp3["House Style"].fillna("Onbekend")
data_exp3 = data_exp3.fillna(0)

data_exp3_encoded = pd.get_dummies(data_exp3, columns=["House Style"])

scaler_exp3 = StandardScaler()
data_exp3_scaled = scaler_exp3.fit_transform(data_exp3_encoded)

kmeans_exp3 = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_exp3 = kmeans_exp3.fit_predict(data_exp3_scaled)

inertia_exp3 = kmeans_exp3.inertia_
silhouette_exp3 = silhouette_score(data_exp3_scaled, labels_exp3)

print("Inertia / WSS:", inertia_exp3)
print("Silhouette score:", silhouette_exp3)

logger.info("Experiment 3 uitgevoerd.")
logger.info(f"Experiment 3 - features: {features_exp3}")
logger.info("Experiment 3 - categorische features: House Style")
logger.info("Experiment 3 - geschaald: Ja")
logger.info("Experiment 3 - hyperparameters: n_clusters=4, random_state=42, n_init=10")
logger.info(f"Experiment 3 - inertia: {inertia_exp3:.2f}")
logger.info(f"Experiment 3 - silhouette score: {silhouette_exp3:.4f}")

experiment_resultaten.append({
    "Experiment": "Experiment 3 - andere categorische feature",
    "Features": ", ".join(features_exp3),
    "Categorische features": "House Style",
    "Geschaald": "Ja",
    "n_clusters": 4,
    "n_init": 10,
    "Inertia": inertia_exp3,
    "Silhouette score": silhouette_exp3
})

### Experiment 4 – Meer features, ander aantal clusters en andere hyperparameter

In dit experiment combineren we:
- meer features,
- een ander aantal clusters,
- een andere waarde voor `n_init`.

Zo testen we een duidelijk andere configuratie dan de eerste run.

In [ ]:
features_exp4 = ["Overall Qual", "Gr Liv Area", "Total Bsmt SF", "Year Built", "Neighborhood"]
data_exp4 = df[features_exp4].copy()
data_exp4["Neighborhood"] = data_exp4["Neighborhood"].fillna("Onbekend")
data_exp4 = data_exp4.fillna(0)

data_exp4_encoded = pd.get_dummies(data_exp4, columns=["Neighborhood"])

scaler_exp4 = StandardScaler()
data_exp4_scaled = scaler_exp4.fit_transform(data_exp4_encoded)

kmeans_exp4 = KMeans(n_clusters=5, random_state=42, n_init=20)
labels_exp4 = kmeans_exp4.fit_predict(data_exp4_scaled)

inertia_exp4 = kmeans_exp4.inertia_
silhouette_exp4 = silhouette_score(data_exp4_scaled, labels_exp4)

print("Inertia / WSS:", inertia_exp4)
print("Silhouette score:", silhouette_exp4)

logger.info("Experiment 4 uitgevoerd.")
logger.info(f"Experiment 4 - features: {features_exp4}")
logger.info("Experiment 4 - categorische features: Neighborhood")
logger.info("Experiment 4 - geschaald: Ja")
logger.info("Experiment 4 - hyperparameters: n_clusters=5, random_state=42, n_init=20")
logger.info(f"Experiment 4 - inertia: {inertia_exp4:.2f}")
logger.info(f"Experiment 4 - silhouette score: {silhouette_exp4:.4f}")

experiment_resultaten.append({
    "Experiment": "Experiment 4 - meer features en ander k",
    "Features": ", ".join(features_exp4),
    "Categorische features": "Neighborhood",
    "Geschaald": "Ja",
    "n_clusters": 5,
    "n_init": 20,
    "Inertia": inertia_exp4,
    "Silhouette score": silhouette_exp4
})

### Vergelijking van alle experimenten

Hier zetten we alle resultaten naast elkaar.  
Voor de eindkeuze kijken we vooral naar de **silhouette score** en naar de duidelijkheid van de gebruikte features.

In [ ]:
vergelijking_df = pd.DataFrame(experiment_resultaten)
vergelijking_df = vergelijking_df.sort_values("Silhouette score", ascending=False)

vergelijking_df

In [ ]:
beste_experiment = vergelijking_df.iloc[0]

print("Beste experiment op basis van silhouette score:")
print(beste_experiment[["Experiment", "Silhouette score", "n_clusters"]])

logger.info(f"Beste experiment op basis van silhouette score: {beste_experiment['Experiment']}")

### Korte conclusie

Het beste experiment is het experiment met de hoogste **silhouette score**.  
Dat experiment presteert beter dan de initiële run, omdat de clusters daarin duidelijker van elkaar gescheiden zijn.

Bij de verantwoording kun je dus uitleggen:
1. welke eerste keuze je maakte,
2. welk resultaat daaruit kwam,
3. welk nieuw experiment je daarna probeerde,
4. en waarom dat nieuwe experiment beter of juist minder goed werkte.